# Heart Disease Prediction Notebook

This notebook implements a machine learning solution for predicting heart disease using ensemble methods. It includes data preprocessing, feature engineering, model training with cross-validation, and submission generation.

## Overview
- **Dataset**: Playground Series S6E2 - Heart Disease
- **Task**: Binary classification to predict heart disease presence
- **Models**: XGBoost, LightGBM, CatBoost with ensemble averaging
- **Key Features**: Extensive feature engineering including polynomial features, ratios, binning, and encoding

## Import Required Libraries

This cell imports all the necessary Python libraries for data manipulation, visualization, preprocessing, and machine learning.

- **pandas (pd)**: For data manipulation and analysis
- **numpy (np)**: For numerical operations
- **matplotlib.pyplot (plt)**: For plotting (though not heavily used here)
- **sklearn modules**: For preprocessing, pipelines, and transformers
- **LabelEncoder**: To convert categorical target to numerical

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, PolynomialFeatures, KBinsDiscretizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import PolynomialFeatures

## Load Data

Load the training and test datasets from CSV files. Store the test IDs for later submission.

- **train.csv**: Training data with features and target
- **test.csv**: Test data without target
- **submission_id**: Preserve test IDs for submission file

In [3]:
train = pd.read_csv('../train.csv')
test = pd.read_csv('../test.csv')
submission_id = test['id']

## Initial Data Preprocessing

Define the target variable, identify feature columns, encode the categorical target to numerical, and prepare X, y, test data.

- **TARGET**: 'Heart Disease' column
- **features**: All columns except 'id' and target
- **LabelEncoder**: Converts 'Presence'/'Absence' to 0/1
- **X**: Training features
- **y**: Encoded target
- **test**: Test features (drop 'id')

In [4]:
TARGET = 'Heart Disease'
features = [col for col in train.columns if col not in ['id',TARGET]]
print('Orig features:',features)

le = LabelEncoder()
train['Heart Disease'] = le.fit_transform(train[TARGET])

X = train.drop(columns = ['id',TARGET])
test.drop(columns='id', inplace = True)
y = train[TARGET]
print(y[:3])

Orig features: ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']
0    1
1    0
2    0
Name: Heart Disease, dtype: int64


## Feature Engineering Function

This comprehensive function creates numerous engineered features from the raw data:

**Feature Categories:**
- **Polynomial Features**: Degree-2 interactions of continuous variables
- **Ratios & Products**: Domain-inspired combinations like cholesterol/BP ratio
- **Relative Differences**: Normalized by mean values
- **Ranking Features**: Rank-based features for ordinal relationships
- **Nonlinear Transforms**: Log, sqrt, inverse transformations
- **Composite Flags**: Boolean combinations for high-risk indicators
- **Frequency Encoding**: For categorical variables
- **Binning**: Discretization of continuous variables
- **Boolean Combinations**: Logical AND operations
- **One-hot Encoding**: For categorical variables

**Returns:** Transformed dataframe, feature lists for tracking

In [5]:
def transform_features(df):

    cols = df.columns
    categorical_vars = ['Sex','Chest pain type','EKG results','Slope of ST','Thallium']
    cont_vars = ['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression']
    binary_vars = ['FBS over 120','Exercise angina']

    # Polynomial Feature
    poly = PolynomialFeatures(degree=2, include_bias=False)
    poly_features = poly.fit_transform(df[cont_vars])
    poly_df = pd.DataFrame(poly_features, 
                        columns=poly.get_feature_names_out(cont_vars)).drop(columns=[i for i in cont_vars])

    # Ratios and domain-inspired features
    df['chol_bp_ratio'] = df['Cholesterol'] / (df['BP'] + 1e-5)
    df['age_hr_ratio'] = df['Age'] / (df['Max HR'] + 1e-5)
    df['bp_chol_product'] = df['BP'] * df['Cholesterol']
    df['risk_score'] = df['Age'] * df['BP'] * df['Cholesterol'] / df['Max HR']
    df['bp_chol_age_index'] = (df['BP'] * df['Cholesterol']) / df['Age']
    df['stress_index'] = df['ST depression'] * df['Max HR']
    df['angina_risk'] = df['Exercise angina'] * df['Age']

    # Relative Differences
    df['chol_relative'] = df['Cholesterol'] / df['Cholesterol'].mean()
    df['bp_relative'] = df['BP'] / df['BP'].mean()

    # Ranking Features
    df['age_rank'] = df['Age'].rank()
    df['chol_rank'] = df['Cholesterol'].rank()

    # Nonlinear Transforms
    df['log_bp'] = np.log1p(df['BP'])
    df['sqrt_chol'] = np.sqrt(df['Cholesterol'])
    df['inv_hr'] = 1 / (df['Max HR'] + 1e-5)
    df['log_chol'] = np.log1p(df['Cholesterol'])

    # Composite Boolean Flags
    df['high_risk_flag'] = ((df['BP'] > 140) & 
                            (df['Cholesterol'] > 240) & 
                            (df['Age'] > 50)).astype(int)

    # Frequency Encoding for categorical vars
    for col in ['Chest pain type','EKG results','Thallium']:
        freq_map = df[col].value_counts(normalize=True)
        df[col+'_freq'] = df[col].map(freq_map)

    # Binning
    df['age_bin'] = pd.cut(df['Age'], bins=3, labels=[1,2,3]).astype(int)
    df['chol_bin'] = pd.cut(df['Cholesterol'], bins=3, labels=[1,2,3]).astype(int)

    # Boolean combinations
    df['fbs_angina'] = ((df['FBS over 120'] == 1) & (df['Exercise angina'] == 1)).astype(int)
     
   
    # One-hot encoding categorical variables
    one_hot_vars = ['Chest pain type', 'EKG results', 'Slope of ST', 'Thallium']
    df_encoded = pd.get_dummies(df[one_hot_vars], columns=one_hot_vars, drop_first=True)
    

    df.drop(columns=[i for i in one_hot_vars],inplace=True)
    final_df = pd.concat([df,df_encoded, poly_df], axis=1)

    poly_features = poly_df.columns.to_list()
    gen_features  = [i for i in df.columns if i not in cols]
    end_features = df_encoded.columns.to_list()


    return final_df, final_df.columns.to_list(), poly_features, gen_features, end_features

trans_df, all_features, poly_features, gen_features, end_features = transform_features(X)
trans_test, _, _, _, _= transform_features(test)
print('Total_features:', all_features,'\nlen:',len(all_features))

Total_features: ['Age', 'Sex', 'BP', 'Cholesterol', 'FBS over 120', 'Max HR', 'Exercise angina', 'ST depression', 'Number of vessels fluro', 'chol_bp_ratio', 'age_hr_ratio', 'bp_chol_product', 'risk_score', 'bp_chol_age_index', 'stress_index', 'angina_risk', 'chol_relative', 'bp_relative', 'age_rank', 'chol_rank', 'log_bp', 'sqrt_chol', 'inv_hr', 'log_chol', 'high_risk_flag', 'Chest pain type_freq', 'EKG results_freq', 'Thallium_freq', 'age_bin', 'chol_bin', 'fbs_angina', 'Chest pain type_2', 'Chest pain type_3', 'Chest pain type_4', 'EKG results_1', 'EKG results_2', 'Slope of ST_2', 'Slope of ST_3', 'Thallium_6', 'Thallium_7', 'Age^2', 'Age BP', 'Age Cholesterol', 'Age Max HR', 'Age ST depression', 'BP^2', 'BP Cholesterol', 'BP Max HR', 'BP ST depression', 'Cholesterol^2', 'Cholesterol Max HR', 'Cholesterol ST depression', 'Max HR^2', 'Max HR ST depression', 'ST depression^2'] 
len: 55


## Feature Scaling

Apply MinMaxScaler to normalize all features to [0,1] range. This is important for gradient boosting models.

- **fit_transform** on training data
- **transform** on test data (using training statistics)

In [6]:
# Scaling df
mms = MinMaxScaler()
final_df = mms.fit_transform(trans_df)
final_test = mms.transform(trans_test)

## Convert to DataFrame

Convert the scaled numpy arrays back to pandas DataFrames with proper column names for model training.

In [7]:
X = pd.DataFrame(final_df, columns=all_features)
final_test = pd.DataFrame(final_test, columns=all_features)

## Exploratory Data Analysis

Placeholder for EDA section. (Note: Original notebook has minimal EDA)

`EDA`

## Model Training with Cross-Validation

Train XGBoost, LightGBM, and CatBoost models using 5-fold stratified cross-validation. Test three different hyperparameter configurations.

**Hyperparameter Cases:**
- **Case 1**: learning_rate=0.1, n_estimators=300, max_depth=4, subsample=0.9
- **Case 2**: learning_rate=0.05, n_estimators=500, max_depth=6, subsample=0.8  
- **Case 3**: learning_rate=0.03, n_estimators=800, max_depth=8, subsample=0.7

**Process:**
- Stratified KFold to maintain class balance
- Out-of-fold predictions for each model
- ROC-AUC scoring for evaluation
- Ensemble predictions (average of three models)

**Output:** OOF ROC-AUC scores for each model and ensemble

In [8]:
import time

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings("ignore") 

# Define parameter cases
common_params_case1 = {
    "learning_rate": 0.1,
    "n_estimators": 300,
    "max_depth": 4,
    "subsample": 0.9,
    "random_state": 42
}

common_params_case2 = {
    "learning_rate": 0.05,
    "n_estimators": 500,
    "max_depth": 6,
    "subsample": 0.8,
    "random_state": 42
}

common_params_case3 = {
    "learning_rate": 0.03,
    "n_estimators": 800,
    "max_depth": 8,
    "subsample": 0.7,
    "random_state": 42
}

param_cases = {
    "case1": common_params_case1,
    "case2": common_params_case2,
    "case3": common_params_case3
}

# Stratified KFold setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Loop through all cases
for case_name, params in param_cases.items():
    print("___________________________")
    print(f"\nRunning {case_name}...")

    # Initialize OOF arrays
    oof_xgb = np.zeros(len(X))
    oof_lgb = np.zeros(len(X))
    oof_cat = np.zeros(len(X))

    # Cross-validation loop
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        print("___________________________")
        print(f"---------FOLD {fold + 1}---------")

        # XGBoost
        start = time.time()
        model_xgb = XGBClassifier(eval_metric="logloss", **params)
        model_xgb.fit(X_train, y_train)
        oof_xgb[val_idx] = model_xgb.predict_proba(X_val)[:,1]
        print(f"XGB {fold} time: {time.time() - start:.2f} sec")


        # LightGBM
        start = time.time()
        model_lgb = LGBMClassifier(**params,verbose=-1)
        model_lgb.fit(X_train, y_train)
        oof_lgb[val_idx] = model_lgb.predict_proba(X_val)[:,1]
        print(f"LGBM {fold} time: {time.time() - start:.2f} sec")

        # CatBoost
        start = time.time()
        model_cat = CatBoostClassifier(verbose=0, **params)
        model_cat.fit(X_train, y_train)
        oof_cat[val_idx] = model_cat.predict_proba(X_val)[:,1]
        print(f"CATboost {fold} time: {time.time() - start:.2f} sec")
        print("___________________________")

        # Compute OOF ROC-AUC
    auc_xgb = roc_auc_score(y, oof_xgb)
    auc_lgb = roc_auc_score(y, oof_lgb)
    auc_cat = roc_auc_score(y, oof_cat)
    auc_ensemble = roc_auc_score(y, (oof_xgb + oof_lgb + oof_cat) / 3)


    print(f"OOF ROC-AUC XGB ({case_name}): {auc_xgb:.4f}")
    print(f"OOF ROC-AUC LGBM ({case_name}): {auc_lgb:.4f}")
    print(f"OOF ROC-AUC CatBoost ({case_name}): {auc_cat:.4f}")
    print(f"OOF ROC-AUC Ensemble ({case_name}): {auc_ensemble:.4f}")

    print("___________________________")

___________________________

Running case1...
___________________________
---------FOLD 0---------
XGB 0 time: 13.52 sec
LGBM 0 time: 10.95 sec
CATboost 0 time: 15.57 sec
___________________________
___________________________
---------FOLD 1---------
XGB 1 time: 7.91 sec
LGBM 1 time: 7.89 sec
CATboost 1 time: 15.97 sec
___________________________
___________________________
---------FOLD 2---------
XGB 2 time: 7.99 sec
LGBM 2 time: 7.49 sec
CATboost 2 time: 16.44 sec
___________________________
___________________________
---------FOLD 3---------
XGB 3 time: 8.17 sec
LGBM 3 time: 8.18 sec
CATboost 3 time: 15.80 sec
___________________________
___________________________
---------FOLD 4---------
XGB 4 time: 8.93 sec
LGBM 4 time: 7.41 sec
CATboost 4 time: 15.33 sec
___________________________
OOF ROC-AUC XGB (case1): 0.9551
OOF ROC-AUC LGBM (case1): 0.9551
OOF ROC-AUC CatBoost (case1): 0.9550
OOF ROC-AUC Ensemble (case1): 0.9552
___________________________
___________________________

R

## Train Final Models

Train the best performing models (based on CV results) on the full training dataset for final predictions.

- **XGBoost**: Using case1 parameters
- **LightGBM**: Using case1 parameters  
- **CatBoost**: Using case2 parameters

**Timing:** Records training time for each model

In [9]:
import time
import pandas as pd
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Define models with params
best_xgb = XGBClassifier(eval_metric="logloss", verbosity=0, **common_params_case1)
best_lgb = LGBMClassifier(verbose=-1, **common_params_case1)
best_cat = CatBoostClassifier(verbose=0, **common_params_case2)

# Timing + training
start = time.time()
best_xgb.fit(X, y)
print(f"time xgb {time.time() - start:.2f} sec")

start = time.time()
best_lgb.fit(X, y)
print(f"time lgb {time.time() - start:.2f} sec")

start = time.time()
best_cat.fit(X, y)
print(f"time cat {time.time() - start:.2f} sec")

time xgb 8.13 sec
time lgb 8.48 sec
time cat 39.17 sec


## Generate Predictions

Generate probability predictions on test data using each trained model, then create ensemble predictions by averaging the three models' outputs.

In [10]:
# Predict hard labels (0/1)
pred_xgb = best_xgb.predict_proba(final_test)[:,1]
pred_lgb = best_lgb.predict_proba(final_test)[:,1]
pred_cat = best_cat.predict_proba(final_test)[:,1]

# Ensemble: majority vote
pred_ensemble = (pred_xgb + pred_lgb + pred_cat )/3
pred_ensemble

array([0.95501832, 0.01022208, 0.98618517, ..., 0.09402989, 0.29408439,
       0.04502378], shape=(270000,))

## Apply Threshold

Convert ensemble probabilities to binary predictions using a threshold of 0.6. Values above 0.6 are classified as positive (heart disease present).

In [11]:
pred_ensemble = np.where(pred_ensemble>0.6, 1, 0)
pred_ensemble

array([1, 0, 1, ..., 0, 0, 0], shape=(270000,))

## Create Submission File

Create the submission DataFrame with 'id' and 'Heart Disease' columns, then save to CSV file named 'submission_2.csv'.

In [ ]:
# Save submission file
submission = pd.DataFrame({
    "id": submission_id,   # or np.arange(len(X_test)) if no explicit IDs
    "Heart Disease": pred_ensemble
})
submission.to_csv("submission.csv", index=False)

In [14]:
submission

,id,Heart Disease
0,630000,1
1,630001,0
2,630002,1
3,630003,0
4,630004,0
...,...,...
269995,899995,0
269996,899996,1
269997,899997,0
269998,899998,0
